# NCAA March Madness Modeling & Prediction

This notebook handles:
1. Creating a matchup-level training dataset from team features.
2. Training a predictive model (XGBoost).
3. Evaluating performance via Log Loss.
4. Generating predictions for the tournament.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
import os

DATA_PATH = './march-machine-learning-mania-2026/'

# Load precomputed features and tournament results
team_features = pd.read_csv('m_team_features.csv')
tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))

print("Data loaded.")

Data loaded.


## 1. Constructing the Training Dataset

We need to create rows for each historical tournament game. To avoid bias, we define Team 1 as the team with the lower ID.

In [2]:
def prepare_training_data(results_df, features_df):
    training_rows = []
    
    for i, row in results_df.iterrows():
        season = row['Season']
        t1 = row['WTeamID'] if row['WTeamID'] < row['LTeamID'] else row['LTeamID']
        t2 = row['LTeamID'] if row['WTeamID'] < row['LTeamID'] else row['WTeamID']
        
        label = 1 if row['WTeamID'] == t1 else 0
        
        # Get features for both teams
        f1 = features_df[(features_df['Season'] == season) & (features_df['TeamID'] == t1)].iloc[0]
        f2 = features_df[(features_df['Season'] == season) & (features_df['TeamID'] == t2)].iloc[0]
        
        # Create difference features
        diff_features = {
            'Season': season,
            'SeedDiff': f1['SeedInt'] - f2['SeedInt'],
            'OffEffDiff': f1['OffEff'] - f2['OffEff'],
            'RankDiff': f1['AvgRank'] - f2['AvgRank'],
            'eFGDiff': f1['eFG'] - f2['eFG'],
            'ScoreDiff': f1['Score'] - f2['Score'],
            'label': label
        }
        training_rows.append(diff_features)
        
    return pd.DataFrame(training_rows)

train_df = prepare_training_data(tourney_results, team_features)
print(f"Training dataset created with {len(train_df)} games.")
train_df.head()

Training dataset created with 1449 games.


,Season,SeedDiff,OffEffDiff,RankDiff,eFGDiff,ScoreDiff,label
0,2003,0.0,2.256412,-1.062500,0.014867,1.593103,0
1,2003,-15.0,8.061088,-150.448529,0.023279,17.421182,1
2,2003,3.0,3.510905,14.294118,0.017069,1.448276,1
3,2003,5.0,-4.352474,24.952206,0.001197,0.102403,1
4,2003,-1.0,-2.209891,-13.906250,-0.010679,2.082759,1


## 2. Model Training & Evaluation

We'll use K-Fold cross-validation and evaluate using Log Loss.

In [3]:
X = train_df.drop(['Season', 'label'], axis=1)
y = train_df['label']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    score = log_loss(y_val, preds)
    scores.append(score)

print(f"Average CV Log Loss: {np.mean(scores):.4f}")

d:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [04:57:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [04:57:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [04:57:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [04:57:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\anaconda\Lib\site-pac

Average CV Log Loss: 0.7542


## 3. Generate Submission Features

For the final submission, we apply this model to every possible matchup in the requested seasons.

In [4]:
def predict_matchup(season, t1, t2, model, features_df):
    # Ensure t1 < t2 for consistent features if needed, but here we just need to be consistent with training
    f1 = features_df[(features_df['Season'] == season) & (features_df['TeamID'] == t1)].iloc[0]
    f2 = features_df[(features_df['Season'] == season) & (features_df['TeamID'] == t2)].iloc[0]
    
    features = pd.DataFrame([{
        'SeedDiff': f1['SeedInt'] - f2['SeedInt'],
        'OffEffDiff': f1['OffEff'] - f2['OffEff'],
        'RankDiff': f1['AvgRank'] - f2['AvgRank'],
        'eFGDiff': f1['eFG'] - f2['eFG'],
        'ScoreDiff': f1['Score'] - f2['Score']
    }])
    
    return model.predict_proba(features)[:, 1][0]

print("Inference function ready.")

Inference function ready.


## 4. Generate Final Submission

We load the sample submission, extract the IDs, and predict the probabilities.

In [ ]:
submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))

# Pre-train the model on the full training set for final submission
final_model = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False)
final_model.fit(X, y)

def get_submission_pred(row):
    # ID format: Season_T1_T2
    parts = row['ID'].split('_')
    season = int(parts[0])
    t1 = int(parts[1])
    t2 = int(parts[2])
    
    return predict_matchup(season, t1, t2, final_model, team_features)

submission['Pred'] = submission.apply(get_submission_pred, axis=1)
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully!")
submission.head()